# Notebook 02b — Download HealthChat-11K Conversation Text

HealthChat-11K's local file contains **metadata only** (taxonomy codes, specialty labels, turn counts).
The actual conversation text lives in two source datasets:

| Source | Count | HuggingFace dataset | Access |
|---|---|---|---|
| lmsys | 20,388 | `lmsys/lmsys-chat-1m` | **Gated — requires approval** |
| wildchat | 12,634 | `allenai/WildChat` | Public |

## What this notebook does

1. Streams `allenai/WildChat` and extracts the 12,634 conversations referenced by HealthChat-11K.
2. Streams `lmsys/lmsys-chat-1m` (if access is available) and extracts the 20,388 conversations.
3. Saves the extracted conversation text to `data/raw/health11k_text/`.
4. Reports how many were successfully matched.

## Prerequisites

- You must be logged in to HuggingFace: `huggingface-cli login`
- For lmsys, you must have **accepted the dataset terms** at:
  https://huggingface.co/datasets/lmsys/lmsys-chat-1m
  (click 'Agree and access repository')
- WildChat is public, no approval needed.

## Output

- `data/raw/health11k_text/wildchat_conversations.parquet` — matched WildChat turns
- `data/raw/health11k_text/lmsys_conversations.parquet` — matched LMSYS turns (if accessible)
- `data/raw/health11k_text/download_report.json` — match statistics

In [ ]:
import os
import json
import pandas as pd
import pyarrow.ipc as ipc
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

RAW_DIR = Path('../data/raw')
OUT_DIR = RAW_DIR / 'health11k_text'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUT_DIR.resolve())

## 1. Load the HealthChat-11K ID sets

In [ ]:
with open(RAW_DIR / 'health11k/train/data-00000-of-00001.arrow', 'rb') as f:
    reader = ipc.open_stream(f)
    tbl = reader.read_all()
meta_df = tbl.to_pandas()

wildchat_ids = set(meta_df[meta_df['dataset_source'] == 'wildchat']['conversation_id'].tolist())
lmsys_ids    = set(meta_df[meta_df['dataset_source'] == 'lmsys']['conversation_id'].tolist())

print(f'WildChat IDs to fetch : {len(wildchat_ids):,}')
print(f'LMSYS IDs to fetch    : {len(lmsys_ids):,}')
print()
print('Sample WildChat IDs:', list(wildchat_ids)[:3])
print('Sample LMSYS IDs:', list(lmsys_ids)[:3])

## 2. Download WildChat conversations

Streams `allenai/WildChat` and filters for the 12,634 IDs we need.
This avoids downloading the full 650K dataset.

In [ ]:
wildchat_out = OUT_DIR / 'wildchat_conversations.parquet'

if wildchat_out.exists():
    print(f'Already exists: {wildchat_out} — skipping download.')
    print(f'Delete the file to re-download.')
    wc_df = pd.read_parquet(wildchat_out)
    print(f'Loaded {len(wc_df):,} rows, {wc_df["conversation_id"].nunique():,} conversations.')
else:
    print('Streaming allenai/WildChat ...')
    print('(This streams the full dataset to find your 12,634 IDs — may take 10-20 minutes)')
    print()

    wc_records = []
    remaining = wildchat_ids.copy()

    ds = load_dataset('allenai/WildChat', split='train', streaming=True)
    pbar = tqdm(total=len(remaining), desc='WildChat matched')

    for row in ds:
        conv_id = row.get('conversation_id')
        if conv_id in remaining:
            conversation = row.get('conversation', [])
            # Expand each turn into a row
            for turn_idx, turn in enumerate(conversation):
                wc_records.append({
                    'conversation_id': conv_id,
                    'turn_index': turn_idx,
                    'role': turn.get('role', ''),
                    'content': turn.get('content', ''),
                    'language': row.get('language', ''),
                    'model': row.get('model', ''),
                    'toxic': turn.get('toxic', False),
                    'redacted': turn.get('redacted', False),
                })
            remaining.discard(conv_id)
            pbar.update(1)
            if not remaining:
                print('\nAll WildChat IDs found — stopping early.')
                break

    pbar.close()

    wc_df = pd.DataFrame(wc_records)
    wc_df.to_parquet(wildchat_out, index=False)
    print(f'\nSaved: {wildchat_out}')
    print(f'Rows: {len(wc_df):,}')
    print(f'Unique conversations matched: {wc_df["conversation_id"].nunique():,} / {len(wildchat_ids):,}')
    if remaining:
        print(f'IDs not found in WildChat: {len(remaining):,}')
        print('Sample missing:', list(remaining)[:5])

## 3. Download LMSYS conversations

**Requires gated access.** If your account has not been approved at
https://huggingface.co/datasets/lmsys/lmsys-chat-1m, this cell will fail with an access error.

If you don't have access yet, skip this cell — WildChat alone adds 12,634 conversations with text.

In [ ]:
lmsys_out = OUT_DIR / 'lmsys_conversations.parquet'

if lmsys_out.exists():
    print(f'Already exists: {lmsys_out} — skipping download.')
    print(f'Delete the file to re-download.')
    lm_df = pd.read_parquet(lmsys_out)
    print(f'Loaded {len(lm_df):,} rows, {lm_df["conversation_id"].nunique():,} conversations.')
else:
    print('Checking access to lmsys/lmsys-chat-1m ...')
    try:
        ds_lm = load_dataset('lmsys/lmsys-chat-1m', split='train', streaming=True)
        # If we get here, we have access
        print('Access confirmed. Streaming lmsys-chat-1m ...')
        print('(This streams ~1M conversations to find your 20,388 IDs — may take 30-60 minutes)')
        print()

        lm_records = []
        remaining = lmsys_ids.copy()

        pbar = tqdm(total=len(remaining), desc='LMSYS matched')

        for row in ds_lm:
            conv_id = row.get('conversation_id')
            if conv_id in remaining:
                conversation = row.get('conversation', [])
                for turn_idx, turn in enumerate(conversation):
                    lm_records.append({
                        'conversation_id': conv_id,
                        'turn_index': turn_idx,
                        'role': turn.get('role', ''),
                        'content': turn.get('content', ''),
                        'model': row.get('model', ''),
                        'language': row.get('language', ''),
                        'redacted': turn.get('redacted', False) if isinstance(turn, dict) else False,
                    })
                remaining.discard(conv_id)
                pbar.update(1)
                if not remaining:
                    print('\nAll LMSYS IDs found — stopping early.')
                    break

        pbar.close()

        lm_df = pd.DataFrame(lm_records)
        lm_df.to_parquet(lmsys_out, index=False)
        print(f'\nSaved: {lmsys_out}')
        print(f'Rows: {len(lm_df):,}')
        print(f'Unique conversations matched: {lm_df["conversation_id"].nunique():,} / {len(lmsys_ids):,}')
        if remaining:
            print(f'IDs not found in LMSYS: {len(remaining):,}')

    except Exception as e:
        if 'gated' in str(e).lower() or 'access' in str(e).lower():
            print('\n*** ACCESS DENIED ***')
            print('You need to request access to lmsys/lmsys-chat-1m at:')
            print('  https://huggingface.co/datasets/lmsys/lmsys-chat-1m')
            print()
            print('Steps:')
            print('  1. Go to the URL above')
            print('  2. Click "Agree and access repository"')
            print('  3. Wait for approval (usually instant)')
            print('  4. Re-run this cell')
            print()
            print('The pipeline will work with WildChat only (12,634 conversations) until then.')
        else:
            print(f'Unexpected error: {e}')
            raise

## 4. Download Report

In [ ]:
report = {
    'wildchat': {
        'ids_needed': len(wildchat_ids),
        'downloaded': 0,
        'status': 'not_run',
    },
    'lmsys': {
        'ids_needed': len(lmsys_ids),
        'downloaded': 0,
        'status': 'not_run',
    }
}

if wildchat_out.exists():
    wc_df = pd.read_parquet(wildchat_out)
    report['wildchat']['downloaded'] = int(wc_df['conversation_id'].nunique())
    report['wildchat']['turns'] = int(len(wc_df))
    report['wildchat']['status'] = 'complete' if report['wildchat']['downloaded'] == len(wildchat_ids) else 'partial'

if lmsys_out.exists():
    lm_df = pd.read_parquet(lmsys_out)
    report['lmsys']['downloaded'] = int(lm_df['conversation_id'].nunique())
    report['lmsys']['turns'] = int(len(lm_df))
    report['lmsys']['status'] = 'complete' if report['lmsys']['downloaded'] == len(lmsys_ids) else 'partial'
else:
    report['lmsys']['status'] = 'pending_access'

report_path = OUT_DIR / 'download_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print('=== DOWNLOAD REPORT ===')
print(json.dumps(report, indent=2))
print()
total_downloaded = report['wildchat']['downloaded'] + report['lmsys']['downloaded']
total_needed = len(wildchat_ids) + len(lmsys_ids)
print(f'Total conversations with text: {total_downloaded:,} / {total_needed:,} ({100*total_downloaded/total_needed:.1f}%)')

## What to run next

Once this notebook completes, run **`07_rebuild_health11k.ipynb`** to:
1. Join the downloaded conversation text with HealthChat-11K metadata
2. Rebuild the structural harmonization with real utterances for HealthChat
3. Re-run semantic + conversational harmonization
4. Re-run validation and export

Or if you only have WildChat text for now, the rebuild notebook handles partial data gracefully.